# 05 · Orchestration

So far LangGraph built the graph for you: `create_agent` is a two-node loop where the **model** decides
every step. Many systems are better as a **workflow**, where **code** decides the steps and the model
fills in the parts that need language. LangGraph's `StateGraph` lets you draw either, or mix them.

**You will learn**

- how to build a graph yourself: a typed state, nodes, edges and conditional edges
- when to use a workflow, an agent, or an agent inside a workflow
- how to build a loop whose limit is enforced in code, not in the prompt
- the research agent's design, which the capstone builds on

In [1]:
# Imports and setup shared by the cells below.
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

from agent_contracts.harness import verify_model
from lg_agents.llm import make_llm

verify_model()  # stops with the fix if Ollama or the shared model is missing
llm = make_llm()

## A first graph: routing ShipFast messages

A `StateGraph` needs three things. A **state** schema (a `TypedDict`): the shared record every node reads
and updates. **Nodes**: plain functions that take the state and return the keys they change. **Edges**:
what runs next, fixed or chosen by a function.

ShipFast's first decision is a good fit for code: triage the message (notebook 01), and if it is out of
scope, decline without ever starting the specialist agent. First, the pieces the nodes will use: the
triage instructions and the specialist agent from notebook 03.

In [2]:
from langchain.agents import create_agent

from agent_contracts.shipfast import tools as shipfast
from agent_contracts.shipfast.schema import Triage
from lg_agents.prompts import SPECIALIST
from lg_agents.tools import shipfast_tools

# The triage instructions: what each intent means, and when order_id should be null.
TRIAGE_PROMPT = ("Classify the customer's message for ShipFast support. intent: order_status, refund_request, "
                 "complaint, or other (anything not about orders, deliveries, refunds or complaints). "
                 "order_id: the order ID such as A1001 if given, else null.")

# The tool-using agent from notebook 03. The graph will call it only for in-scope messages.
specialist = create_agent(llm, shipfast_tools(), system_prompt=SPECIALIST)

Next, the **state**: a `TypedDict` listing what the nodes share. `total=False` means a key may be missing until a node sets it, so the graph can start with just the message.

In [3]:
from typing import TypedDict


class Support(TypedDict, total=False):
    message: str   # the customer's message: the input
    triage: Triage  # set by the triage node
    reply: str     # set by decline or resolve: the output

Now the **nodes** and the routing function. Each node takes the state and returns only the keys it changes. `route` returns the name of the node that runs next.

In [4]:
def triage(state: Support) -> Support:
    # Model call: read the message and return an intent and an order ID (structured output, as in notebook 01).
    return {"triage": llm.with_structured_output(Triage).invoke(
        [SystemMessage(TRIAGE_PROMPT), HumanMessage(state["message"])])}


def decline(state: Support) -> Support:  # no model call at all
    return {"reply": "Sorry, I can only help with orders, deliveries, refunds and complaints."}


def resolve(state: Support) -> Support:
    # Agent call: the specialist chooses its own tools and writes the reply.
    result = specialist.invoke({"messages": [HumanMessage(state["message"])]})
    return {"reply": result["messages"][-1].content}


def route(state: Support) -> str:
    # The decision lives in code: out-of-scope messages go to decline, everything else to resolve.
    return "decline" if state["triage"].intent == "other" else "resolve"

Now wire the nodes together with edges and compile the graph. `add_conditional_edges` calls `route` after `triage` and follows the edge it names.

In [5]:
g = StateGraph(Support)
g.add_node("triage", triage)
g.add_node("decline", decline)
g.add_node("resolve", resolve)
g.add_edge(START, "triage")  # fixed edge: every run starts with triage
g.add_conditional_edges("triage", route, ["decline", "resolve"])  # the list names the nodes route may return
g.add_edge("decline", END)
g.add_edge("resolve", END)
support = g.compile()  # checks the wiring and returns a graph you can invoke

Print the edges to check the wiring: three fixed edges, and two conditional ones out of `triage`.

In [6]:
print(f"{'from':9} -> {'to':9} kind")
for e in support.get_graph().edges:
    print(f"{e.source:9} -> {e.target:9} {'conditional' if e.conditional else 'fixed'}")

from      -> to        kind
__start__ -> triage    fixed
triage    -> decline   conditional
triage    -> resolve   conditional
decline   -> __end__   fixed
resolve   -> __end__   fixed


Now run the graph on two messages, one out of scope and one about an order. For each you see what triage decided, which tools ran, and the reply.

In [7]:
for message in ("Can you write me a poem about cats?", "Where is order A9999?"):
    shipfast.reset()  # clear the tool-call log, so only this message shows up
    out = support.invoke({"message": message})  # out is the final state: message, triage and reply
    print("message:", message)
    print("  intent:  ", out["triage"].intent)
    print("  order_id:", out["triage"].order_id)
    print("  tools:   ", ", ".join(shipfast.tool_calls()) or "none")
    print("  reply:   ", out["reply"].strip()[:150])
    print()

message: Can you write me a poem about cats?
  intent:   other
  order_id: None
  tools:    none
  reply:    Sorry, I can only help with orders, deliveries, refunds and complaints.



message: Where is order A9999?
  intent:   order_status
  order_id: A9999
  tools:    lookup_order
  reply:    I couldn't find any order with the ID A9999. Could you please double-check the order ID or provide more details?



The poem never reached the specialist: a fixed edge in code handled it, cheaply and predictably. The
order question went to the agent, which chose its own tools. That mix, **a workflow outside and an agent
inside**, is the pattern the crash course recommends: structure where you can, autonomy where you must.

## The research agent as a workflow

The research agent (`contracts/agent_contracts/research/SPEC.md`) must plan, research with at most 5 tool
calls, draft a cited answer, critique it, and revise at most twice. Most of that is a fixed sequence, so it
is a workflow. Only the research step is open-ended (which searches? is a calculation needed?), so that
step is a small agent. It lives in `lg_agents/research_flow.py`:

In [8]:
# The docstring of lg_agents/research_flow.py, shown because it draws the graph this notebook explains.
import lg_agents.research_flow as rf

print(rf.__doc__)

The research agent as a LangGraph workflow (research SPEC.md section 1), taught in notebook 05.

Workflow outside, agent inside: fixed steps (plan, draft, critique, revise) with a small tool-using agent
for the research step. The revision limit is enforced by the graph's edges, in code, never by the prompt.
The tool-call budget (5) is enforced by the shared tools.

    plan -> research -> draft -> critique --pass--> END
                                   ^   \--revise--> revise --(revisions < 2)--> critique
                                                        \--(revisions == 2)--> END



Now build the flow and list its edges, in the same format as the first graph.

In [9]:
flow = rf.build_research_flow(llm)
print(f"{'from':9} -> {'to':9} kind")
for e in flow.get_graph().edges:
    print(f"{e.source:9} -> {e.target:9} {'conditional' if e.conditional else 'fixed'}")

from      -> to        kind
__start__ -> plan      fixed
critique  -> __end__   conditional
critique  -> revise    conditional
draft     -> critique  fixed
plan      -> research  fixed
research  -> draft     fixed
revise    -> __end__   conditional
revise    -> critique  conditional


Each node is one step, with its own instructions (`rf.RESEARCHER`, `rf.WRITER`, `rf.CRITIC`). The
writer and critic use structured output, so the draft comes back as an answer plus citations, and the
critique as a verdict plus issues, the shared `Critique` model from `agent_contracts`.

## Running it, step by step

Question Q01 asks how many employees Veltara Aquatech has. The fixtures contain the current figure (350,
in a 2026 article) and a stale one (200, in a 2023 article), and an employees search returns both. Watch
each node's update:

In [10]:
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

questions = {q["id"]: q for q in load_questions()}
q = questions["Q01_headcount"]
research.reset()  # clear the tool-call log and budget

# The flow's state, built up from each node's update as it streams past.
state = {"question_id": q["id"], "question": q["question"]}
for update in flow.stream(state, stream_mode="updates"):  # one update per node, as soon as it finishes
    for node, change in update.items():
        state.update(change)
        print(f"[{node}]")
        for key, value in change.items():  # the keys this node changed, cut to 160 characters
            text = " ".join(str(value).split())
            print(f"  {key}: {text[:160]}{' ...' if len(text) > 160 else ''}")
        print()

[plan]
  plan: ["Search for Veltara Aquatech's official website to find any publicly available information about their employee count.", 'Look for recent press releases, inves ...
  revisions: 0
  critiques: []



[research]
  findings: Tool results: {"query": "Veltara Aquatech employee count", "results": [{"id": "F02", "title": "Veltara grows team to 350 employees ahead of Cebu factory opening ...



[draft]
  answer: Veltara Aquatech currently has 350 employees, as reported in the article published on 2026-03-18 [F02].
  citations: ['F02']



[critique]
  critiques: [{'verdict': 'pass', 'issues': []}]



Now turn the final state into the shared report format and score it with the shared scorer, a plain text check against the question's expected facts.

In [11]:
report = rf.to_report(state)
score = score_report(report, q)

print("tool calls:", ", ".join(research.tool_calls()) or "none")
print("passed:    ", score["passed"])
for name, value in score["scores"].items():
    print(f"  {name + ':':20}", value)
print("problems:  ", score["problems"] or "none")

tool calls: web_search
passed:     True
  fact_recall:         1.0
  facts_missing:       []
  contradiction_free:  True
  citations_valid:     True
problems:   none


Look closely at what came back. The draft has the right figure, 350, with one citation, and never
mentions the outdated 200. The critic, a model call told to send back any draft that mentions an outdated
figure, passed it, and the shared scorer, a plain text check, agrees (`contradiction_free: True`). The two
do not always agree on a run like this, though: a model checking a model can miss things a deterministic
check does not, which is why ShipFast and the research agent are scored in code, never by another model call.

## The loop, and its limit

When the critique's verdict is `revise`, the graph goes to `revise` and back to `critique`. The critic is
not useless: given a draft that relies on the stale figure outright, with the real findings from that run,
it does send it back:

In [12]:
# A draft that relies on the stale figure (200) outright, checked against the real findings from the run above.
critique = rf.critique_draft(llm, q["question"], state["findings"],
                             "Veltara Aquatech has 200 employees [F03].", ["F03"])

print("verdict:", critique.verdict)
print("issues:")
for issue in critique.issues:
    print("  -", issue)

verdict: revise
issues:
  - The draft answer cites F03, which reports 200 employees, but the most recent and supported figure is 350 employees from F02. The question asks for the current number of employees, and F02 is the latest source. The draft answer is therefore incorrect and should be revised to reflect the most recent data.


What stops a critic that is never satisfied? Not the prompt. The two routing functions decide every
transition, and `route_after_revise` ends the run after the second revision whatever the critic would
say. These are plain functions, so you can check the rule directly:

In [13]:
# Each case is a situation, the state it produces, and the routing function that reads it.
cases = [
    ("critic says pass", {"critiques": [{"verdict": "pass"}]}, rf.route_after_critique),
    ("critic says revise", {"critiques": [{"verdict": "revise"}]}, rf.route_after_critique),
    ("1 revision so far", {"revisions": 1}, rf.route_after_revise),
    ("2 revisions so far", {"revisions": 2}, rf.route_after_revise),
]
print(f"{'routing function':22} {'situation':20} goes to")  # "__end__" means the run stops
for situation, state_in, rule in cases:
    print(f"{rule.__name__:22} {situation:20} {rule(state_in)}")

routing function       situation            goes to
route_after_critique   critic says pass     __end__
route_after_critique   critic says revise   revise
route_after_revise     1 revision so far    critique
route_after_revise     2 revisions so far   __end__


So a run can only end in the three shapes the spec allows: `[pass]`, `[revise, pass]`, or
`[revise, revise]` after two revisions. `score_report` checks that shape for every report.

## Workflow or agent?

| | Workflow (`StateGraph`) | Agent (`create_agent`) |
|---|---|---|
| Who decides the next step | your code | the model |
| Predictability | high: the same path every time | lower: it can skip or repeat steps |
| Good for | known procedures, limits that must hold, cheap routing | open-ended investigation |
| In this tutorial | ShipFast routing, the research loop | ShipFast's specialist, the research step |

## How the others do this

- **CrewAI** draws workflows as a **Flow**: methods decorated `@start` and `@listen` run in order, and a
  `@router` method returns the name of the next step, like a conditional edge. Agents and crews run inside
  flow steps, the same "workflow outside, agent inside" shape.
- **Strands** builds multi-step systems from agents: `GraphBuilder` connects agents as nodes with
  conditional edges, and `Swarm` lets agents hand off to each other. Plain code steps are less central
  than in LangGraph.

## Takeaways

- A `StateGraph` is a typed state, nodes that update it, and edges, some decided by functions.
- Use code for the steps you know and an agent for the steps you do not; mix them freely.
- Put limits in the edges. A loop bounded by a routing function cannot run away.
- Next: **06 · Guardrails and approval**.